# Exploratory Data Analysis — xG Dataset

Loads shot events from StatsBomb open data across several competitions, applies the feature engineering pipeline, and analyses the resulting dataset.

**Competitions loaded:** La Liga 2015/16, Champions League 2018/19, Premier League 2015/16, FIFA World Cup 2022.

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from mplsoccer import Pitch
from statsbombpy import sb

from src.features import create_xg_features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## 1. Data Loading

In [2]:
COMPETITIONS = [
    {'competition_id': 11, 'season_id': 27,  'label': 'La Liga 2015/16'},
    {'competition_id': 16, 'season_id': 4,   'label': 'Champions League 2018/19'},
    {'competition_id': 2,  'season_id': 27,  'label': 'Premier League 2015/16'},
    {'competition_id': 43, 'season_id': 106, 'label': 'World Cup 2022'},
]

def load_events_for_competition(competition_id, season_id):
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    frames = []
    for match_id in matches['match_id']:
        events = sb.events(match_id=match_id)
        events['match_id'] = match_id
        frames.append(events)
    return pd.concat(frames, ignore_index=True)

all_events = []
for comp in COMPETITIONS:
    print(f"Loading {comp['label']}…")
    df = load_events_for_competition(comp['competition_id'], comp['season_id'])
    df['competition'] = comp['label']
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
print(f"\nTotal events: {len(events_df):,}")

Loading La Liga 2015/16…
Loading Champions League 2018/19…
Loading Premier League 2015/16…
Loading World Cup 2022…

Total events: 2,846,929


## 2. Feature Engineering

In [4]:
xg_df = create_xg_features(events_df)

# Attach competition label back onto shots
comp_map = events_df.drop_duplicates('match_id')[['match_id', 'competition']]
xg_df = xg_df.merge(comp_map, on='match_id', how='left')

# Exclude penalties from the main analysis (modelled separately)
shots = xg_df[xg_df['is_penalty'] == 0].copy()

print(f"Shots (excl. penalties): {len(shots):,}")
print(f"Goals:                   {shots['is_goal'].sum():,} ({shots['is_goal'].mean():.1%})")
shots.head(3)

KeyError: 'shot'

## 3. Dataset Overview

In [ ]:
print(f"Shape: {shots.shape}")
print(f"\nShots per competition:")
print(shots.groupby('competition')['is_goal'].agg(['count', 'sum', 'mean'])
      .rename(columns={'count': 'shots', 'sum': 'goals', 'mean': 'conversion'})
      .round(3))

In [ ]:
# Missing value rates
null_rates = (shots.isnull().sum() / len(shots)).sort_values(ascending=False)
null_rates = null_rates[null_rates > 0]

fig, ax = plt.subplots(figsize=(7, 4))
null_rates.plot.barh(ax=ax)
ax.set_xlabel('Missing rate')
ax.set_title('Feature missingness')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

## 4. Class Balance

In [ ]:
outcome_counts = shots['is_goal'].value_counts().rename({0: 'No goal', 1: 'Goal'})

fig, ax = plt.subplots(figsize=(4, 3))
outcome_counts.plot.bar(ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='white', rot=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Class balance')
ax.set_ylabel('Shots')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

print(f"Imbalance ratio  →  {outcome_counts[0]/outcome_counts[1]:.1f}:1 (no goal : goal)")

## 5. Spatial Analysis

In [ ]:
# Shot map — all shots coloured by outcome
pitch = Pitch(pitch_type='statsbomb', pitch_color='#1a1a2e', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

no_goals = shots[shots['is_goal'] == 0]
goals    = shots[shots['is_goal'] == 1]

pitch.scatter(no_goals['shot_x'], no_goals['shot_y'], ax=ax,
              s=12, color='#4C72B0', alpha=0.25, label='No goal')
pitch.scatter(goals['shot_x'], goals['shot_y'], ax=ax,
              s=25, color='#DD8452', alpha=0.7, label='Goal')

ax.legend(loc='lower left', framealpha=0.4, labelcolor='white',
          facecolor='#1a1a2e', edgecolor='white')
ax.set_title('Shot locations', color='white', fontsize=13, pad=10)
fig.patch.set_facecolor('#1a1a2e')
plt.tight_layout()
plt.show()

In [ ]:
# Goal rate heatmap across pitch zones
pitch = Pitch(pitch_type='statsbomb', line_zorder=2)
fig, ax = pitch.draw(figsize=(10, 7))

bin_statistic = pitch.bin_statistic(
    shots['shot_x'], shots['shot_y'], shots['is_goal'],
    statistic='mean', bins=(18, 12)
)
hm = pitch.heatmap(bin_statistic, ax=ax, cmap='YlOrRd', edgecolors='#22312b')
cbar = fig.colorbar(hm, ax=ax, shrink=0.6)
cbar.set_label('Goal rate', fontsize=10)
ax.set_title('Goal rate by pitch zone', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Distance and Angle Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for outcome, label, color in [(0, 'No goal', '#4C72B0'), (1, 'Goal', '#DD8452')]:
    subset = shots[shots['is_goal'] == outcome]
    axes[0].hist(subset['distance_to_goal'].dropna(), bins=40, alpha=0.55,
                 label=label, color=color, density=True, edgecolor='none')
    axes[1].hist(np.degrees(subset['shot_angle'].dropna()), bins=40, alpha=0.55,
                 label=label, color=color, density=True, edgecolor='none')

axes[0].set_xlabel('Distance to goal (SB units)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distance distribution')
axes[0].legend()

axes[1].set_xlabel('Shot angle (degrees)')
axes[1].set_title('Angle distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Goal rate in distance buckets
shots['distance_bin'] = pd.cut(shots['distance_to_goal'], bins=[0, 6, 12, 18, 24, 30, 40, 60])
rate_by_dist = shots.groupby('distance_bin', observed=False)['is_goal'].agg(['mean', 'count'])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

rate_by_dist['mean'].plot.bar(ax=ax1, color='#DD8452', alpha=0.85, edgecolor='white')
ax2.plot(range(len(rate_by_dist)), rate_by_dist['count'], 'o--', color='#4C72B0',
         linewidth=1.5, markersize=5, label='Shot count')

ax1.set_ylabel('Goal rate')
ax1.set_xlabel('Distance to goal (SB units)')
ax2.set_ylabel('Shot count', color='#4C72B0')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.set_title('Goal rate by distance')
ax1.set_xticklabels([str(b) for b in rate_by_dist.index], rotation=30, ha='right')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 7. Goal Rate by Categorical Features

In [ ]:
def goal_rate_bar(ax, col, title, min_shots=30):
    stats = (shots.groupby(col)['is_goal']
             .agg(['mean', 'count'])
             .query('count >= @min_shots')
             .sort_values('mean', ascending=True))
    bars = ax.barh(stats.index, stats['mean'], color='#4C72B0', edgecolor='white')
    for bar, count in zip(bars, stats['count']):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f'n={count:,}', va='center', fontsize=8, color='0.4')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(title)
    ax.set_xlabel('Goal rate')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
goal_rate_bar(axes[0], 'shot_body_part',  'Body part')
goal_rate_bar(axes[1], 'shot_technique',  'Technique')
goal_rate_bar(axes[2], 'previous_event_type', 'Previous event type')
plt.tight_layout()
plt.show()

In [ ]:
# Boolean flags
flag_cols = [
    ('in_penalty_area',      'In penalty area'),
    ('in_six_yard_box',      'In six-yard box'),
    ('shot_first_time',      'First time'),
    ('shot_under_pressure',  'Under pressure'),
    ('is_late_game',         'Late game (≥75 min)'),
    ('is_extra_time',        'Extra time'),
]

flag_df = pd.DataFrame([
    {
        'Feature': label,
        'Yes (goal rate)': shots[shots[col] == 1]['is_goal'].mean(),
        'No (goal rate)':  shots[shots[col] == 0]['is_goal'].mean(),
        'Yes (count)': int((shots[col] == 1).sum()),
    }
    for col, label in flag_cols
]).set_index('Feature')

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(flag_df))
width = 0.35
ax.bar(x - width/2, flag_df['Yes (goal rate)'], width, label='Yes', color='#DD8452', edgecolor='white')
ax.bar(x + width/2, flag_df['No (goal rate)'],  width, label='No',  color='#4C72B0', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(flag_df.index, rotation=20, ha='right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel('Goal rate')
ax.set_title('Goal rate by binary flag')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Freeze Frame Coverage and Features

In [ ]:
freeze_cols = [
    'num_defenders_in_frame', 'num_teammates_in_frame',
    'distance_to_nearest_defender', 'num_defenders_between_shot_and_goal',
    'goalkeeper_distance_to_goal', 'goalkeeper_distance_to_shooter',
]

coverage = shots[freeze_cols].notna().mean()
print("Freeze frame feature coverage:")
print(coverage.map(lambda x: f'{x:.1%}'))

has_freeze = shots['num_defenders_in_frame'].notna()
print(f"\nShots with freeze frame data: {has_freeze.sum():,} / {len(shots):,} ({has_freeze.mean():.1%})")

In [ ]:
freeze_shots = shots[has_freeze]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, freeze_cols):
    for outcome, label, color in [(0, 'No goal', '#4C72B0'), (1, 'Goal', '#DD8452')]:
        subset = freeze_shots[freeze_shots['is_goal'] == outcome][col].dropna()
        ax.hist(subset, bins=25, alpha=0.55, label=label, color=color,
                density=True, edgecolor='none')
    ax.set_title(col.replace('_', ' '))
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Freeze frame feature distributions (goal vs no goal)', y=1.01)
plt.tight_layout()
plt.show()

## 9. Numeric Feature Summary Statistics

In [ ]:
numeric_features = [
    'distance_to_goal', 'shot_angle', 'centrality',
    'num_defenders_in_frame', 'num_teammates_in_frame',
    'distance_to_nearest_defender', 'num_defenders_between_shot_and_goal',
    'goalkeeper_distance_to_goal', 'goalkeeper_distance_to_shooter',
    'previous_event_distance',
]

summary = (
    shots.groupby('is_goal')[numeric_features]
    .mean()
    .T
    .rename(columns={0: 'No goal (mean)', 1: 'Goal (mean)'})
)
summary['Δ (goal − no goal)'] = summary['Goal (mean)'] - summary['No goal (mean)']
summary.round(3)

## 10. Correlation Heatmap

In [ ]:
corr_features = [
    'is_goal', 'distance_to_goal', 'shot_angle', 'centrality',
    'in_penalty_area', 'in_six_yard_box',
    'shot_first_time', 'shot_under_pressure',
    'num_defenders_in_frame', 'distance_to_nearest_defender',
    'num_defenders_between_shot_and_goal',
    'goalkeeper_distance_to_goal', 'goalkeeper_distance_to_shooter',
    'previous_event_was_pass', 'previous_event_was_carry', 'previous_event_distance',
    'is_late_game', 'is_extra_time',
]

corr = shots[corr_features].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.4, square=True, cbar_kws={'shrink': 0.6},
)
ax.set_title('Feature correlation matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Correlation with `is_goal` (ranked)

In [ ]:
goal_corr = corr['is_goal'].drop('is_goal').sort_values(key=abs, ascending=True)

colors = ['#DD8452' if v > 0 else '#4C72B0' for v in goal_corr]
fig, ax = plt.subplots(figsize=(7, 6))
goal_corr.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with is_goal')
ax.set_title('Feature correlations with goal outcome')
plt.tight_layout()
plt.show()

## 12. Penalty Analysis (separate)

In [ ]:
penalties = xg_df[xg_df['is_penalty'] == 1]
print(f"Penalties: {len(penalties):,}")
print(f"Penalty conversion rate: {penalties['is_goal'].mean():.1%}")
print(f"Open-play conversion rate: {shots['is_goal'].mean():.1%}")